# Group 1 — Can a cheap surrogate replace an expensive sensitivity study?

**AI Capstone project — finding important turbulence-model parameters with explainable AI**

---

## The engineering problem

Computational Fluid Dynamics (CFD) predicts turbulent flows using a *turbulence
model*. The model used here, **SST $k$–$\omega$**, contains nine empirical
constants. Nobody knows their exact values; each is only known to lie inside an
interval. Different choices inside those intervals give different predicted
flows, so an engineer who wants a trustworthy prediction needs to know **which of
the nine constants actually matter**.

The standard way to answer this is a *Sobol sensitivity analysis*: sample the
parameter space many times, run a CFD simulation for every sample, and decompose
the variance of the output. For this dataset that took **1408 CFD simulations per
flow case** — weeks of supercomputer time.

## Your question

> Could we have reached the same conclusion with far fewer simulations, by
> training a machine-learning model on a small subset and asking *it* which
> parameters matter?

This is what Group 1 investigates. You will:

1. train surrogate models that predict a CFD output from the nine parameters;
2. measure how their accuracy and their recovered parameter ranking degrade as
   you take away training data;
3. compute sensitivity indices *from the surrogate* and compare them against the
   indices computed from the full CFD study.

The Sobol indices from the full study are provided to you as a **benchmark**. You
never have to compute them from CFD — that part is already done.

---

---

## How to use this notebook

This is the **skeleton of your project**, not a worked example. The explanatory
text, the data loading and the figures that show you the provided data are
given. Everything else is yours.

Look for `# TODO` blocks. Each states a task and names the variables that later
cells expect. They deliberately do **not** give you the exact function call:
[`RESOURCES.md`](../RESOURCES.md) lists every method you need with a link to its
documentation and a runnable example, and choosing your own settings — then
being able to say why — is part of what you are assessed on.

Sections are labelled **MUST**, **SHOULD** or **COULD**:

* **MUST** — the project. Every group does these.
* **SHOULD** — where your group goes deeper. Agree with your supervisor in the
  project plan which of these you commit to.
* **COULD** — optional and unscaffolded. If you take one on, you design it.

The last few sections are not analysis at all: dividing the work, risks, ethics,
and a list of questions nobody has answered yet. Read them in week one, not in
week nine.

**You are six or seven people.** The MUST sections are perhaps a week of work
for a pair. They are the starting point, not the project. Split into
sub-teams early — there is a suggested division near the end — and remember
that two people doing the same thing twice is the most common way a capstone
group wastes a month.

If you disagree with a choice made here, change it and say why when you
present. That is a better outcome than following it.

In [ ]:
# --- Setup --------------------------------------------------------------------
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone


def find_project_root(marker="src/capstone_tools.py"):
    """Locate the top of the cloned repository.

    Walks up from the current working directory, so the notebook works whether
    Jupyter was started in notebooks/, in the project root, or anywhere else
    inside the clone. Nothing here depends on where you put the folder.
    """
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / marker).exists():
            return base
    raise FileNotFoundError(
        "Could not find the project root. Open this notebook from inside the "
        "cloned repository, so that a 'src/capstone_tools.py' exists in one of "
        "the parent directories of " + str(Path.cwd())
    )


ROOT = find_project_root()
sys.path.insert(0, str(ROOT / "src"))
import capstone_tools as ct

warnings.filterwarnings("ignore", category=FutureWarning)
ct.set_plot_style()

RANDOM_STATE = 0          # fixed so that your results are reproducible
FIGDIR = ROOT / "figures"
FIGDIR.mkdir(exist_ok=True)

print("project root  :", ROOT)
print("data directory:", ct.DATA_DIR)
print("parameters:", ", ".join(ct.PARAMETER_NAMES))

## 1. The nine turbulence-model parameters

Each parameter was varied inside the interval shown below. These intervals come
from the turbulence-modelling literature: they are the range of values different
authors consider defensible.

In [ ]:
info = ct.load_parameter_info()
info

## 2. The data

We start with **turbulent channel flow at $Re_\tau = 640$** — flow between two
parallel walls, the most studied configuration in all of turbulence research.

For each of the 1408 simulations the CFD solver produced profiles of several
quantities across the channel, sampled at 40 positions. Two of those quantities
matter here:

| symbol | meaning | why it matters |
|---|---|---|
| $p'_{\mathrm{rms}}$ | root-mean-square pressure fluctuation | drives vibration and fatigue of structures in the flow |
| $k$ | turbulent kinetic energy | the central quantity of the turbulence model itself |

From these 40-point profiles we have derived six **scalar outputs** — single
numbers per simulation. These are what you will predict, because a single number
is far easier to model and to explain than a whole profile.

In [ ]:
CASE, RE_TAU = "channel", 640

data = ct.load_scalar_outputs(CASE, RE_TAU)
coords = ct.load_coordinates(CASE, RE_TAU)

print(f"{data.shape[0]} simulations x {len(ct.PARAMETER_NAMES)} parameters")
print("scalar outputs:", ", ".join(ct.SCALAR_OUTPUTS))
data.head()

### What the simulations actually look like

Before modelling anything, look at the data. The grey lines below are individual
CFD simulations: same flow, same solver, only the nine turbulence-model constants
differ. The spread between them *is* the uncertainty we are trying to attribute
to individual parameters.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, var in zip(axes, ["p_rms", "k"]):
    prof = ct.load_profiles(CASE, var, RE_TAU)
    values = prof[[c for c in prof.columns if c.startswith("point_")]].to_numpy()
    y = coords["y_plus"].to_numpy()
    for row in values[::20]:
        ax.plot(row, y, color="0.7", lw=0.6, alpha=0.6)
    ax.plot(values.mean(axis=0), y, color="crimson", lw=2.4, label="ensemble mean")
    ax.set_xlabel(ct.VARIABLE_LABELS[var])
    ax.set_ylabel(r"$y^+$ (distance from the wall)")
    ax.set_yscale("log")
    ax.set_title(f"{ct.VARIABLE_LABELS[var]} — {len(values)} simulations")
    ax.legend()
fig.suptitle(f"Spread caused by turbulence-model parameters ({ct.case_label(CASE, RE_TAU)})")
fig.tight_layout()
fig.savefig(FIGDIR / "G1_01_data_spread.png")
plt.show()

## 3. The benchmark: what the full CFD study found

The Sobol index $S_i$ of parameter $i$ is the fraction of the output variance
that is explained by varying that parameter alone. The indices sum to
(approximately) one, so you can read them as percentages of the variance.

Two kinds are provided:

* **first-order $S_1$** — the effect of a parameter *on its own*;
* **total-order $S_T$** — its effect *including* everything it does in
  combination with other parameters.

This is the answer your workflow is trying to reproduce from much less data.

In [ ]:
bench = ct.load_sobol_scalar(CASE, RE_TAU)
QOI = "p_rms_peak"          # the output we model first

# Permutation importance and SHAP both include whatever a parameter does in
# combination with the others, so the TOTAL-order index is the like-for-like
# comparison. We use ST as the primary reference throughout and keep S1 for the
# interaction discussion in section 12.
BENCHMARK = "ST"

bench_q = (bench[bench.output == QOI]
           .set_index("parameter")
           .reindex(ct.PARAMETER_NAMES))

fig, ax = plt.subplots(figsize=(9, 4))
order = bench_q[BENCHMARK].sort_values(ascending=False).index
idx = np.arange(len(order))
ax.bar(idx - 0.2, bench_q.loc[order, "ST"], width=0.4,
       yerr=bench_q.loc[order, "ST_conf"], capsize=3,
       color=[ct.COLOURS[p] for p in order], label="$S_T$ (total order)")
ax.bar(idx + 0.2, bench_q.loc[order, "S1"], width=0.4,
       yerr=bench_q.loc[order, "S1_conf"], capsize=3,
       color=[ct.COLOURS[p] for p in order], alpha=0.45, label="$S_1$ (first order)")
ax.set_xticks(idx)
ax.set_xticklabels([ct.PRETTY[p] for p in order])
ax.set_ylabel("Sobol index")
ax.set_title(f"Benchmark from all 1408 CFD simulations — {QOI}")
ax.legend()
fig.tight_layout()
fig.savefig(FIGDIR / "G1_02_benchmark.png")
plt.show()

print(f"sum of S1 = {bench_q.S1.sum():.3f}   sum of ST = {bench_q.ST.sum():.3f}")
print("A sum close to 1 means the parameters act almost independently, so S1 and")
print("ST nearly coincide here - but ST is the one we compare against.")
print("\nbenchmark ranking (by ST):")
for rank, p in enumerate(order, 1):
    print(f"  {rank}. {p:12s} ST = {bench_q.loc[p, 'ST']:.4f}   S1 = {bench_q.loc[p, 'S1']:.4f}")

Look at the ranking you just printed before going further.

* How is the variance distributed across the nine parameters? Is there a clean
  break between parameters that matter and parameters that do not, and if so,
  where does it fall?
* Which parts of this ranking would you expect to be easy to recover from a
  small number of simulations, and which hard? Write your prediction down now,
  and check it against section 8 later.

That distribution is the thing the rest of the project is trying to recover
from much less data, so it is worth a few minutes of attention.

---
## 4. MUST — a surrogate model and an honest error estimate

### 4a. Splitting the data: why a random split would mislead you

The 1408 simulations are **not** 1408 independent random samples. They come from
a *Saltelli design*, built in 128 blocks of 11. Within a block, the samples
differ from one another in **only one of the nine parameters** — that structure is
exactly what makes the Sobol estimator work.

The consequence for us: if you split rows at random, almost every test
simulation has a near-twin in the training set, differing in one coordinate out
of nine. Any model will predict those nearly perfectly, and your test score will
flatter you.

The dataset therefore carries a `design_group` column. Keep a whole block on one
side of the split. Let us measure how much difference this makes.

In [ ]:
# ==========================================================================
# TODO - compare a random split with a grouped split
#
# Find out for yourselves how much a naive random split flatters the model.
#
# Fit the same model twice on the same data: once with an ordinary random split,
# once keeping each design_group together. Report the test score from each, and
# make the comparison fair - the same model, the same test fraction, the same
# seed.
#
# Keep the two numbers. They are a result, not a technicality, and they show you
# understood the data before you modelled it.
#
# Hints:
#   * the dataset's structure is explained in DATA_DICTIONARY.md section 3
#   * RESOURCES.md section 2 lists the splitters that respect groups
#
# Later cells expect:
#   X, y, groups : the feature matrix, the target, and the design-group labels
# ==========================================================================


# your code here

### 4b. The surrogate model

We use a Random Forest: a collection of regression trees, each fitted to a
different bootstrap sample of the data, averaged together. It needs almost no
tuning, handles non-linear responses, and is the standard first choice for a
tabular problem of this size.

In [ ]:
# ==========================================================================
# TODO - fit the surrogate and measure how good it is
#
# Make the grouped split you will use for the rest of the notebook, fit a
# surrogate, and report RMSE, normalised RMSE and R2 on the test set.
#
# Choose the model settings yourselves and be ready to justify them: how many
# trees, how deep, and how you know that choice is not doing the work for you.
# Section 14 asks you to check that very question, so make the choice
# deliberately now rather than accepting a default you cannot defend.
#
# Hints:
#   * RESOURCES.md sections 1 and 3
#   * normalised RMSE is RMSE divided by the spread of the test target - it needs no units, which makes it the one to quote in a talk
#   * a small evaluate() helper will save you repeating this later
#
# Later cells expect:
#   X_train, X_test, y_train, y_test : the grouped split
#   train_idx, test_idx              : the corresponding index arrays
#   forest                           : the fitted surrogate
# ==========================================================================


# your code here

In [ ]:
# ==========================================================================
# TODO - draw the parity plot
#
# Scatter the CFD values against your predictions on the test set, with a
# diagonal for reference and the score in the title.
#
# This is the figure that persuades an audience your surrogate works, so make it
# readable from the back of a room: square axes, equal ranges, a visible
# diagonal. Look at where the points depart from the line - is the error uniform,
# or is the model worse in a particular part of the range?
#
# Hints:
#   * save your figures to FIGDIR so the final presentation is a copy job
# ==========================================================================


# your code here

---
## 5. MUST — explaining the surrogate with permutation feature importance

The surrogate is accurate, but it is a black box: 300 trees tell us nothing
directly about which parameter matters.

**Permutation feature importance** answers that by breaking one parameter at a
time. Shuffle the values of a single column in the *test* set, so the parameter
keeps its distribution but loses its relationship to the output, and re-score the
model. If the score collapses, the model relied on that parameter; if nothing
happens, it did not.

We shuffle several times (`n_repeats`) because each shuffle is random, which also
gives us a spread to quote as an error bar.

In [ ]:
# ==========================================================================
# TODO - rank the parameters with permutation feature importance
#
# Apply permutation feature importance to your fitted surrogate and collect the
# result into a tidy table sorted from most to least important.
#
# Two decisions are yours, and both matter:
#
#   * which data you measure it on, and why;
#   * how many repeats you need before the numbers stop moving - find out rather
#     than guessing.
#
# Hints:
#   * RESOURCES.md section 4, including a worked example
#   * the routine reports a spread as well as a mean - you will want it later
#
# Later cells expect:
#   importance : DataFrame with columns ['parameter', 'importance', 'spread'],
#                sorted by importance descending
# ==========================================================================


# your code here

### Comparing the explanation with the benchmark

Permutation importance is measured in *drop in $R^2$*, while a Sobol index is a
*fraction of variance*. The two are not on the same scale, so comparing their raw
values is meaningless — what we compare is the **ranking**, and the shape of the
profile across parameters.

In [ ]:
# ==========================================================================
# TODO - compare your ranking with the benchmark
#
# Put your importances next to the benchmark Sobol indices.
#
# Before you plot, work out what the two quantities actually measure and whether
# they can share an axis. If they cannot, say so in the figure rather than
# quietly rescaling one of them. Then print both rankings so you can read them
# off.
#
# Hints:
#   * keep a fixed parameter order and a fixed colour per parameter (ct.COLOURS) so every figure in your presentation reads the same way
#
# Later cells expect:
#   merged : DataFrame indexed by parameter, with at least the columns
#            'importance', 'S1', 'ST', and 'benchmark' (= merged[BENCHMARK])
# ==========================================================================


# your code here

---
## 6. SHOULD — put a number on the agreement

Comparing two bar charts by eye is not a result. Three standard measures:

* **Spearman rank correlation** over all nine parameters — but beware: seven of
  the nine have benchmark indices near zero, so their *relative* order is
  essentially noise, and this number is dominated by that noise. Expect it to
  look mediocre and to stop improving with more data.
* **Top-$k$ overlap** — how many of the benchmark's $k$ most important
  parameters appear in the AI method's top $k$. This is the measure an engineer
  cares about, because the point of the exercise is to decide which parameters
  to keep.
* **Kendall's $\tau$** — like Spearman, but based on counting pairwise
  disagreements.

In [ ]:
# ==========================================================================
# TODO - turn the comparison into a number
#
# Write a function that compares an importance vector against a benchmark vector
# and returns several agreement measures - at least a rank correlation and the
# overlap of the top k parameters for a few values of k.
#
# Then apply it, and compare the numbers against your own impression from the two
# bar charts. If a measure disagrees with your eyes, decide which of the two is
# misleading you and why. Look at the benchmark values of the five least
# important parameters before you answer.
#
# Whichever measure you end up quoting, you need to be able to defend it as the
# right one for the engineering question.
#
# Hints:
#   * RESOURCES.md section 5
#
# Later cells expect:
#   rank_agreement(scores, reference, ks=(2, 3, 4)) -> dict
#       used in several later sections, so keep this signature
# ==========================================================================


# your code here

## 7. SHOULD — are the parameters really separated?

An importance score without an uncertainty is not interpretable. Two sources of
spread matter, and they are different things:

1. the **shuffling** spread, which `permutation_importance` already returns;
2. the spread caused by **which simulations landed in the training set**, which
   you get by repeating the whole fit on several different grouped splits.

The second is usually much larger — and it is the one that tells you whether a
conclusion would survive if you had run a different set of simulations.

In [ ]:
# ==========================================================================
# TODO - put error bars on the importance scores
#
# A single importance number means little on its own. Repeat the whole
# fit-and-explain procedure over several different grouped splits and report the
# spread across repeats.
#
# There are two different sources of variability here and they are not the same
# size. Identify both, decide which one your error bars should show, and say why.
#
# Then state which parameters are separated by more than that spread. Those are
# the only ones you can defend as "more important than" one another.
#
# Hints:
#   * sklearn.base.clone gives you a fresh unfitted copy of a model
#   * collect each repeat as a pd.Series indexed by parameter
#
# Later cells expect:
#   summary : DataFrame indexed by parameter with columns ['mean', 'std'],
#             sorted by mean descending
# ==========================================================================


# your code here

---
## 8. SHOULD — how little CFD data is enough? (the core Group 1 result)

Now the central experiment. Train on a *subset* of the simulations and see what
survives. Two rules matter:

* subsample **whole design groups**, never individual rows, for the reason given
  in section 4a;
* repeat each size several times with different random subsets, because a single
  small subset can be lucky or unlucky.

We track two things against the number of simulations: how well the surrogate
predicts, and how well its explanation matches the benchmark.

In [ ]:
# ==========================================================================
# TODO - the core experiment: how little CFD data is enough?
#
# This is the central result of the Group 1 project, and it is a piece of
# experimental design rather than a piece of coding.
#
# Train on progressively smaller subsets and record both how well the surrogate
# predicts and how well its explanation still matches the benchmark.
#
# Before you write the loop, settle three things and write down your reasoning:
#
#   * what exactly you subsample, given how this dataset is built;
#   * how many repeats per size you need for the error bars to mean anything;
#   * what you evaluate on, so that the comparison between sizes is fair.
#
# Get these wrong and the figure will look fine and mean nothing.
#
# Hints:
#   * np.isin(groups, chosen) turns a list of group ids into a boolean mask
#   * time one size before running them all
#   * cache the result to disk - you will want to re-plot without re-fitting
#
# Later cells expect:
#   curve      : tidy DataFrame, one row per (size, repeat), with at least
#                'n_simulations', 'R2' and your agreement measures
#   curve_mean : curve grouped by n_simulations, mean
#   curve_std  : the same, standard deviation
# ==========================================================================


# your code here

In [ ]:
# ==========================================================================
# TODO - plot the learning curves
#
# Two panels: surrogate accuracy against the number of training simulations, and
# agreement with the benchmark against the same axis. Show your error bars, and
# think about whether a linear or logarithmic x-axis tells the truth better.
#
# Then read the figure properly before you move on - the markdown below asks you
# specific questions about it.
# ==========================================================================


# your code here

Read this figure carefully before moving on. The curves do **not** behave the
same way as each other, and explaining why is worth more marks than the figure.

For each curve, answer:

* what does it do as the amount of data increases?
* is that what you predicted in section 3?
* if a curve stops improving, is that a limitation of your method, of the
  measure you chose, or of the data?

At least one of the three will probably surprise you. The explanation is in the
benchmark values themselves, not in your code — go back and look at them before
you decide your workflow has failed.

Whichever measure you put on a slide, be ready to say what it would take for
that number to reach 1.0, and whether reaching 1.0 is even meaningful here.

## 9. SHOULD — is a different surrogate better?

Random Forest was an arbitrary first choice. Compare it with gradient boosting,
which builds trees sequentially, each correcting the previous one's errors.
Both are trained on exactly the same data so the comparison is fair.

In [ ]:
# ==========================================================================
# TODO - is a different surrogate better?
#
# Your first model was an arbitrary choice. Compare it against at least one other
# family on exactly the same training data, and report accuracy, agreement with
# the benchmark, and the top of each ranking in one table.
#
# Then commit to a recommendation: which model would you give an engineer who can
# afford only a few hundred CFD simulations, and what evidence supports that?
# A comparison that ends without a recommendation is only half a result.
#
# Hints:
#   * RESOURCES.md section 1
#   * clone each model before fitting so the comparison is clean
#   * keep optional imports in try/except so the notebook still runs without them
# ==========================================================================


# your code here

## 10. SHOULD — sensitivity indices from the surrogate instead of from CFD

This is the sharpest experiment available in this dataset, and it is only a few
lines of code.

A Sobol analysis needs thousands of model evaluations. CFD cannot afford that —
but a *surrogate* can be evaluated millions of times for free. So:

1. train the surrogate on a few hundred CFD simulations;
2. draw a fresh Saltelli sample of the parameter space;
3. evaluate the **surrogate** on it (not CFD);
4. run the ordinary Sobol estimator on those surrogate predictions.

The result is a set of Sobol indices obtained for the price of a few hundred
simulations, which we can hold directly against the indices from all 1408.

In [ ]:
# ==========================================================================
# TODO - compute Sobol indices from the surrogate instead of from CFD
#
# This is the sharpest experiment available in this dataset.
#
# A Sobol analysis needs thousands of model evaluations, which CFD cannot afford -
# but a surrogate can be evaluated as often as you like. So train a surrogate on a
# few hundred simulations, draw a fresh sample of the parameter space, evaluate
# the *surrogate* on it, and run the ordinary Sobol estimator on those
# predictions.
#
# Do this for a few training budgets and collect the results next to the CFD
# benchmark and your permutation importances in one table.
#
# Two things to think about rather than copy: how large the fresh sample needs to
# be before the indices stop moving, and which index you should be comparing
# against, given what the notebook established earlier.
#
# Hints:
#   * RESOURCES.md section 6; ct.make_salib_problem() builds the problem dict
#   * the sampler and the analyser must agree about second-order terms
#
# Later cells expect:
#   surrogate_sobol(model, problem) -> pd.Series indexed by parameter
#   three_way : DataFrame with the CFD benchmark, your permutation importances,
#               and one column per training budget
# ==========================================================================


# your code here

In [ ]:
# ==========================================================================
# TODO - plot the three-way comparison
#
# A grouped bar chart comparing the CFD benchmark against each surrogate budget.
#
# Underneath, print the top-four parameter set from each source and mark which
# ones match the benchmark exactly. That table is the result; the chart only makes
# it visible.
#
# Look at where the surrogate is accurate and where it is not - the error is not
# spread evenly across the nine parameters, and the pattern is worth a sentence.
# ==========================================================================


# your code here

## 11. SHOULD — the smallest usable number of simulations, and what it saves

The original CFD study reduced the nine parameters to a four-parameter subset
before doing anything expensive with them. So the operational question is: **how many CFD
simulations do you need before the AI workflow identifies that same subset?**

In [ ]:
# ==========================================================================
# TODO - find the smallest usable number of simulations
#
# The original CFD study reduced nine parameters to a four-parameter subset before
# doing anything expensive. So: how many simulations does your workflow need
# before it identifies that same subset?
#
# For each training-set size, repeat several times and count how often you recover
# the subset exactly. Report the success rate against the number of simulations,
# and state the answer the way an engineer would want it.
#
# Then compare this against the top-four curve from section 8, which used
# permutation importance directly. If the two routes need different amounts of
# data, explain why - the reason is in what each method has to get right.
#
# Hints:
#   * compare sets, not lists - the order within the subset does not matter
# ==========================================================================


# your code here

## 12. SHOULD — do the parameters interact?

Compare the first-order index $S_1$ (a parameter acting alone) with the
total-order index $S_T$ (the same parameter including every interaction it takes
part in). The difference $S_T - S_1$ is the part of the variance that parameter
only produces *in combination* with others.

This matters for interpreting your own results: permutation importance and SHAP
both include interaction effects, so if interactions were strong you would expect
them to disagree with $S_1$.

In [ ]:
# ==========================================================================
# TODO - do the parameters interact?
#
# Compare the first-order and total-order indices, print the difference, and look
# at the two column sums.
#
# Then answer the question this raises: given what you find, should you have
# expected permutation importance to agree with the first-order indices, or not?
# And would your conclusions in this notebook have been the same on a problem
# where the sums came out very differently?
#
# Hints:
#   * both columns are already in `merged`
# ==========================================================================


# your code here

---
# Optional extensions (COULD)

Everything above covers the MUST and SHOULD requirements. What follows is
optional. Pick whatever interests your group; you are not expected to do all of
it.

Two Coulds in the project description have no section of their own below,
because they are about the repository rather than about an analysis:

* an **interactive dashboard** for comparing surrogate accuracy and parameter
  importance — `ipywidgets` inside the notebook is enough, and it is a good
  home for the spare person once the three pairs are busy;
* leaving behind a **tidy, documented repository** that could be pointed at a
  new parameter-study dataset with little modification. If you lifted the spine
  into `src/` as suggested above, you are most of the way there already.

Both count, and the second one costs almost nothing if you decide on it early.

## 13. COULD — other kinds of surrogate

Tree ensembles are not the only option. A **Gaussian process** fits a smooth
function and is often the best choice when data is scarce; a small **neural
network** is the most familiar model from an AI course. Both need their inputs
standardised, because the nine parameters have very different magnitudes
(compare $\beta^* \approx 0.09$ with $\alpha_{k2} \approx 1.0$).

In [ ]:
# ==========================================================================
# TODO - optional extension
#
# This section is deliberately unscaffolded. Decide whether your group
# wants it, agree the approach with your supervisor, and build it
# yourselves. The markdown above says what it should achieve.
# ==========================================================================


# your code here

## 14. COULD — does the conclusion survive a change of model settings?

If the recovered ranking flips when you change the number of trees, the ranking
is not a property of the physics — it is an artefact of your model choice. This
is a cheap and very convincing robustness check to show when you present.

In [ ]:
# ==========================================================================
# TODO - optional extension
#
# This section is deliberately unscaffolded. Decide whether your group
# wants it, agree the approach with your supervisor, and build it
# yourselves. The markdown above says what it should achieve.
# ==========================================================================


# your code here

## 15. COULD — one model for many outputs at once

So far we modelled one scalar. A single multi-output forest can predict all six
at once. Is it as good as six separate models, and is it faster?

In [ ]:
# ==========================================================================
# TODO - optional extension
#
# This section is deliberately unscaffolded. Decide whether your group
# wants it, agree the approach with your supervisor, and build it
# yourselves. The markdown above says what it should achieve.
# ==========================================================================


# your code here

## 16. COULD — what would this have saved in CFD time?

Section 11 gave you a number of simulations. Turn it into the number an
engineering manager actually asks for: how much *compute* does the
surrogate-based route save?

You will need a figure for the cost of one CFD simulation — ask your supervisor
for the wall-clock time and core count of a single run in this study. Then
compare the full 1408-simulation analysis against your minimum sample size plus
the (negligible) cost of training and evaluating a surrogate.

State the assumptions on the slide. A saving quoted without them is not a result.

In [ ]:
# ==========================================================================
# TODO - optional extension
#
# This section is deliberately unscaffolded. Decide whether your group
# wants it, agree the approach with your supervisor, and build it
# yourselves. The markdown above says what it should achieve.
# ==========================================================================


# your code here

## 17. COULD — is the parameter ranking the same at other Reynolds numbers?

The dataset contains the same study at four Reynolds numbers. A conclusion that
only holds at one of them is of limited use to an engineer, so this is a strong
robustness result to report.

Here we give the AI workflow a fixed, small budget at each Reynolds number and
ask whether it reproduces the *trend* found by the full CFD study.

In [ ]:
# ==========================================================================
# TODO - optional extension
#
# This section is deliberately unscaffolded. Decide whether your group
# wants it, agree the approach with your supervisor, and build it
# yourselves. The markdown above says what it should achieve.
# ==========================================================================


# your code here

## 18. COULD — a completely different geometry

The annular-flow case is the same turbulence model in a curved geometry (flow in
the gap between a rod and an outer cylinder). Loading it is a one-line change,
which is the point: if your pipeline is written cleanly, a new flow case costs
you nothing.

In [ ]:
# ==========================================================================
# TODO - optional extension
#
# This section is deliberately unscaffolded. Decide whether your group
# wants it, agree the approach with your supervisor, and build it
# yourselves. The markdown above says what it should achieve.
# ==========================================================================


# your code here

---
# Running the project

The sections above are the technical work. These are the things that decide
whether a capstone group finishes well, and they are assessed too.

## Dividing the work

### What blocks what

Most of this notebook parallelises, but not the beginning. Sections 4 to 6
build the objects every later section uses - the split, the fitted surrogate,
the importance table and the agreement function - so **until they exist,
nobody else can start**.

    sections 4-6   the spine: split, surrogate, importance, agreement
      |            ONE pair, first week. Everything else waits on it.
      |
      +-- section 7    error bars                    independent
      +-- section 8    learning curve                independent
      +-- section 9    surrogate comparison          independent
      +-- section 12   interactions                  independent, ~1 hour
      +-- sections 13-15, 18                         independent
      |
      +-- section 10   Sobol from the surrogate      independent
              |
              +-- section 11   minimum budget        needs 10
              +-- section 17   Reynolds sweep        needs 10

    section 16   CFD time saving   needs nothing from the notebook at all

So: a short sequential start, then four or five streams that genuinely run
side by side. Get the spine finished and committed in week one - it is small,
it is the MUST requirements, and every day it is late is a day five people
are blocked.

### Suggested split, once the spine exists

* **Data budget** - sections 7, 8, 11. Owns the learning curve and the
  minimum-sample-size result: the project's headline number.
* **Models** - sections 9, 13, 14. Owns the comparison between surrogate
  families and the robustness checks. Reports which model to use at a budget.
* **Sensitivity** - sections 10, 16, 17, 18. Owns the surrogate-based Sobol
  route, the three-way comparison, and the extensions to other Reynolds
  numbers and the annular case.

With seven people the spare person is best used on shared infrastructure -
caching, the results table, the figures everyone needs - not as a third member
of one pair.


Three rules that save groups a lot of pain:

1. **Lift the spine out of the notebook.** As soon as sections 4-6 work, move
   that code into a small module under `src/` — the split, the fitted model,
   the importance table — so every sub-team imports the same objects instead of
   re-running the notebook top to bottom and quietly getting a different split.
   This single step is what turns a sequential notebook into parallel work.
2. **One shared split, fixed early.** If different sub-teams use different
   train/test splits, none of your numbers can be compared with any other.
   Fix `RANDOM_STATE` and the split once, in writing.
3. **Everything in git, from week one.** Six people editing one notebook is the
   classic failure. Split your work into separate notebooks or modules that
   import from `src/`, and merge deliberately.

## Requirements, and renegotiating them

Your project plan lists the MoSCoW requirements you have agreed with the
supervisor. Keep a table in the repository showing, for each requirement, its
status and the evidence — the figure or the number that demonstrates it. You
will need exactly this table for the final presentation, and filling it in as
you go costs nothing while reconstructing it in week nine costs a day.

If a Must starts looking unreachable, say so early and renegotiate. That is
explicitly allowed, and it is assessed far better than a requirement quietly
dropped. Risk taken knowingly and reported honestly is a good outcome; a
surprise in week nine is not.

## What could go wrong

Write these down in the project plan with a mitigation for each. Some that are
real for this project:

* **The cell that runs for an hour.** Several steps here fit one model per
  output. Time a single output before you loop over all of them, and cache
  results to disk so you never recompute them.
* **Results that cannot be compared.** Different splits, different random
  seeds, different normalisations between sub-teams. Fix them centrally.
* **A finding that disappears on re-run.** If a conclusion depends on one random
  seed, it is not a finding. Repeat anything you intend to present.
* **The deadline shape.** You get three weeks of full-time work and nothing
  before it. Whatever you want to present must exist in a rough form by the end
  of week 2.8, so that week 2.9 is for checking and explaining, not discovering.

## Ethics and societal impact

You will be asked about this, and for this project it is not an afterthought.

The data comes from a study of **turbulence-induced vibration in nuclear fuel
assemblies**: coolant flowing past fuel rods produces unsteady pressure that
shakes them, and over years that causes fretting wear and fatigue. The
turbulence-model parameters you are ranking feed predictions used in safety
assessments.

Questions worth a slide:

* **What is the cost of being wrong in each direction?** Your workflow says
  certain parameters do not matter, so an engineer stops calibrating them. If
  that judgement is wrong, the predicted vibration is wrong. Is the error more
  likely to be optimistic or pessimistic, and which is worse here?
* **Who is accountable** when a cheap surrogate replaces an expensive analysis
  and the cheap one is wrong — the person who built the surrogate, or the person
  who trusted it? What would you have to report alongside a ranking for the
  second person to use it responsibly?
* **Does a confident-looking output deserve confidence?** You will produce
  normalised importances that always sum to one, so they look equally certain
  for every output — including outputs where the surrogate fits badly. What
  would you add to the figure to stop a reader over-reading it?
* **The positive side.** The CFD campaign behind this dataset cost on the order
  of 2 x 10^5 CPU hours. If a surrogate reaches the same conclusion from a
  fraction of the simulations, that is a real saving in energy and emissions as
  well as money. Estimate it — with your assumptions stated — rather than
  asserting it.
* **Automation and judgement.** If this workflow became routine, what expertise
  stops being exercised, and does that matter?

A good answer is specific to this project and admits a trade-off. A generic
paragraph about "AI bias" will not do.

## Open questions

Nobody has answered these on this dataset, including your supervisor. They are
offered because a capstone is more interesting when some of it is real. Each is
a reasonable size for a pair in the full-time weeks, and none has a known
answer, so a careful negative result is a perfectly good outcome.

1. **Can you tell that you do not have enough data, without the benchmark?** In
   a real study nobody gives you the right answer to compare against. Is there
   a signal computable from your own fits alone — disagreement between repeats,
   between models, between bootstrap samples — that tracks whether the ranking
   is right? If there is, that is a genuinely useful result.
2. **Does choosing the next simulations beat choosing them at random?** You
   subsample randomly. Suppose instead you could pick which parameter
   combinations to run next, given what the surrogate already knows. Does a
   targeted choice reach the correct ranking from fewer simulations?
3. **Does the minimum budget transfer?** You measure it at one Reynolds number.
   Is the same budget enough at the others, or must it be re-measured for every
   new flow? An engineer planning a campaign needs to know.
4. **Does the Saltelli structure help or hurt training?** The design is highly
   structured: within a block, points differ in one coordinate. Is that good
   training data, or would a space-filling sample of the same size do better?
   You cannot run new CFD, but you can compare subsets with different structure.
5. **What does the saving really cost?** Count everything the surrogate route
   needs — the simulations, the fitting, the sensitivity analysis, and the human
   time — and compare honestly against the full study.


If you take one on, agree it with your supervisor first, and treat it the way
you would treat a Should: state the question, say how you will answer it, and
report what you actually found rather than what you hoped for.

---
# 19. What you actually hand in

This course has **no written report**. You are assessed on three things:

| when | deliverable |
|---|---|
| before the Christmas break | a **project plan**, agreed with the supervisor |
| week 9 | the **AI solution** — code, results, figures, saved models |
| week 10 | a **final presentation** to your supervisor and the assessors |

So the work below is not material for a report; it is material for a talk and a
clean repository. Plan accordingly: a figure that needs three paragraphs of
explanation is a bad slide.

### For the project plan

Decide early which SHOULD items your group will commit to, and say which ones
you will not do. You have six or seven people — split the work by section, not by
"everyone tries the same model". Natural division for Group 1: one pair on the
learning curve, one pair on surrogate comparison, one pair on the
surrogate-based Sobol route, everyone on interpretation.

### For the week-9 solution

* a repository that runs end to end from a clean clone;
* your notebooks with outputs, so results can be seen without re-running;
* the figures you will actually present, saved as files;
* a short README stating what you did, what you found, and what you did not do.

### For the final presentation

A suggested spine, roughly one slide each:

1. **The question.** Identifying influential turbulence parameters costs 1408
   CFD simulations. Could far fewer do?
2. **The data and the trap.** The Saltelli design, and why a random split
   flatters you. Show the two R² numbers from section 4a — it is a result, not a
   technicality, and it shows you understood the data.
3. **The surrogate works.** Parity plot, with RMSE / nRMSE / R² and your split
   stated on the slide.
4. **The explanation matches the benchmark.** Your ranking next to the Sobol
   ranking, with error bars.
5. **The learning curve.** Your central result. Explain why accuracy and top-2
   overlap behave differently.
6. **The minimum budget.** How few simulations recover the four-parameter
   subset, and what that saves (sections 11 and 16).
7. **Limits.** See below.
8. **Conclusion** in one sentence that answers slide 1.

### Be honest about the limits — it earns marks, it does not lose them

Specific things that are true of this dataset:

* the surrogate systematically under-estimates the small Sobol indices;
* the ranking among the near-zero parameters is noise, and no amount of data
  fixes it;
* finding the top two parameters is almost trivially easy — say so;
* a high R² does not by itself guarantee a correct explanation;
* unless you did section 16, your conclusion was tested at one Reynolds number.